In [0]:
import refrom pathlib import PathS3_PATHS = {    "orders": "s3://fmcg-data-lake/bronze/orders",    "customers": "s3://fmcg-data-lake/bronze/customers",    "gross_price": "s3://fmcg-data-lake/bronze/gross_price",    "products": "s3://fmcg-data-lake/bronze/products",}def stage_s3_folder(    data_source,    catalog,    destination_relative_path,    processed_relative_path=None,    skip_existing=False,):    if data_source not in S3_PATHS:        raise ValueError(f"No S3 path is configured for {data_source!r}")    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):        raise ValueError(f"Invalid Unity Catalog name: {catalog!r}")    if any(part in {".", ".."} for part in Path(destination_relative_path).parts):        raise ValueError("Destination path must stay inside the S3 volume")    if processed_relative_path and any(        part in {".", ".."} for part in Path(processed_relative_path).parts    ):        raise ValueError("Processed path must stay inside the S3 volume")    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`bronze`")    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`bronze`.`s3`")    volume_root = f"/Volumes/{catalog}/bronze/s3"    destination_path = f"{volume_root}/{destination_relative_path.strip('/')}"    dbutils.fs.mkdirs(destination_path)    paths_to_check = [destination_path]    if processed_relative_path:        processed_path = f"{volume_root}/{processed_relative_path.strip('/')}"        dbutils.fs.mkdirs(processed_path)        paths_to_check.append(processed_path)    existing_names = set()    if skip_existing:        for path in paths_to_check:            existing_names.update(                entry.name for entry in dbutils.fs.ls(path) if not entry.isDir()            )    source_root = S3_PATHS[data_source]    all_files = dbutils.fs.ls(source_root)    csv_files = [        entry.path for entry in all_files if entry.path.lower().endswith(".csv")    ]    if not csv_files:        raise RuntimeError(f"S3 path for {data_source!r} contains no CSV files")    staged_count = 0    for source_file in csv_files:        file_name = source_file.rsplit("/", 1)[-1]        if skip_existing and file_name in existing_names:            continue        destination_file = f"{destination_path}/{file_name}"        copied = dbutils.fs.cp(source_file, destination_file, True)        if not copied:            raise RuntimeError(f"Failed to stage {file_name!r} from S3 to the Volume")        staged_count += 1    print(        f"Copied {len(csv_files)} CSV file(s) from S3; staged {staged_count} new file(s) under {destination_path}"    )    return destination_path